# Импорты

In [28]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss
from sklearn.ensemble import BaggingClassifier, GradientBoostingClassifier
from sklearn.base import clone

from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

from scipy import stats

## 1. Классификация ансамблей

### По способу построения базовых моделей

| Тип | Принцип | Примеры |
|-----|---------|---------|
| **Параллельные** | Модели обучаются независимо, результаты агрегируются | Bagging, Random Forest |
| **Последовательные** | Каждая модель учится на ошибках предыдущей | AdaBoost, GBM, XGBoost |
| **Иерархические** | Выходы базовых моделей — вход мета-модели | Stacking, Blending |

- Разные подвыборки **объектов** → Bagging
- Разные подвыборки **признаков** → Random Subspace / Random Forest
- Разные **веса объектов** → AdaBoost, Gradient Boosting

### Сложность

```
Voting/Averaging < Bagging < Random Forest < Blending < Stacking < AdaBoost < GBM < XGBoost
        проще                                                                        сложнее
```


# Загрузка данных

In [29]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# бэггинг

1. Для каждой из `n_estimators` моделей:
   - Сделать **бутстрап-выборку** (случайная выборка с возвращением)
   - Обучить базовую модель на этой выборке
2. Предсказание: **soft voting** — усреднение вероятностей всех моделей

In [30]:
def _bootstrap(X, y, n_samples, rng):
    """Случайная выборка с возвращением."""
    idx = rng.integers(0, len(X), size=n_samples)
    return X[idx], y[idx]

In [31]:
def bagging_fit(X, y, n_estimators=100, max_samples=1.0,
                base_estimator=None, random_state=None):
    """
    n_estimators    — число базовых моделей
    max_samples     — доля объектов для каждого bootstrap
    base_estimator  —  классификатор
    """
    X, y = np.asarray(X), np.asarray(y)

    if base_estimator is None:
        base_estimator = DecisionTreeClassifier()

    rng = np.random.default_rng(random_state)
    n_samples = int(len(X) * max_samples)
    estimators = []

    for _ in range(n_estimators):
        estimator = clone(base_estimator)
        X_boot, y_boot = _bootstrap(X, y, n_samples, rng)
        estimator.fit(X_boot, y_boot)
        estimators.append(estimator)

    return {
        "estimators": estimators,
        "classes":    np.unique(y),
        "n_estimators": n_estimators,
    }

In [32]:
def bagging_predict_proba(model, X):
    """
    Усредняет вероятности всех деревьев
    """
    X = np.asarray(X)
    classes = model["classes"]
    all_proba = []

    for estimator in model["estimators"]:
        if hasattr(estimator, "predict_proba"):
            proba = estimator.predict_proba(X)
        else:
            preds = estimator.predict(X)
            proba = np.zeros((len(X), len(classes)))
            for i, cls in enumerate(classes):
                proba[preds == cls, i] = 1.0
        all_proba.append(proba)

    return np.mean(all_proba, axis=0)

In [33]:
def bagging_predict(model, X):
    proba = bagging_predict_proba(model, X)
    return model["classes"][np.argmax(proba, axis=1)]


def bagging_score(model, X, y):
    return accuracy_score(y, bagging_predict(model, X))

In [34]:
bag_model = bagging_fit(X_train, y_train, n_estimators=100, random_state=42)
print(f"MyBagging   train: {bagging_score(bag_model, X_train, y_train):.4f}  "
      f"test: {bagging_score(bag_model, X_test, y_test):.4f}")

ref = BaggingClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
print(f"sklearn Bag train: {ref.score(X_train, y_train):.4f}  "
      f"test: {ref.score(X_test, y_test):.4f}")

MyBagging   train: 1.0000  test: 0.9474
sklearn Bag train: 1.0000  test: 0.9474


# Voting

In [35]:
def voting_fit(X, y, estimators):
    """
    estimators — список уже сконфигурированных моделей.
    Просто обучаем каждую и сохраняем в словарь.
    """
    trained = []
    for estimator in estimators:
        estimator.fit(X, y)
        trained.append(estimator)

    return {
        "estimators": trained,
        "classes":    np.unique(y),
    }


def voting_predict(model, X):
    """
    Каждая модель голосует за класс.
    Побеждает тот, за кого проголосовало большинство.
    """
    # Собираем предсказания каждой модели: shape = (n_models, n_samples)
    votes = np.array([est.predict(X) for est in model["estimators"]])

    majority, _ = stats.mode(votes, axis=0)
    return majority.flatten()


def voting_score(model, X, y):
    return accuracy_score(y, voting_predict(model, X))

In [36]:
estimators = [
    DecisionTreeClassifier(max_depth=5, random_state=42),
    KNeighborsClassifier(n_neighbors=5),
    LogisticRegression(max_iter=10000),
]

v_model = voting_fit(X_train, y_train, estimators)
print(f"Voting test: {voting_score(v_model, X_test, y_test):.4f}")

Voting test: 0.9415
